# Snapshot spaces and proper orthogonal decomposition

This chapter is also a notebook: use the download toolbar and run its Python cells in order.
The image-compression figures and numerical results are executed at website build time; changing parameters requires running the notebook.
The compression experiment takes about 30 minutes; the rest of the chapter supplies the theory.


POD constructs a space that approximates selected snapshots well in a specified metric.
It answers a different question from solving the [reduced equations](https://feelpp.github.io/course-rom/rom/reduction/galerkin.html) at a new parameter.


**Course route:** recall the SVD geometry and PCA connection, choose samples and a metric, derive the POD projection, and compute modes by SVD or the snapshot correlation matrix. Continuous POD is supplementary reading explaining where sampling weights come from.
The [parameter-sampling section](https://feelpp.github.io/course-rom/rom/reduction/galerkin.html#parameter-sampling) and [definition of Z](https://feelpp.github.io/course-rom/rom/reduction/galerkin.html#basis-matrix) explain how these modes become a predictive reduced basis.

## Recall the SVD: directions, scales and coordinates

For a matrix $Y\in\mathbb R^{n\times m}$, the singular value decomposition is

$$
Y=U\Sigma V^T,\qquad U^TU=I_n,\quad V^TV=I_m.
$$
In the full SVD, $U$ is $n\times n$, $V$ is $m\times m$ and $\Sigma$ is rectangular $n\times m$, with nonnegative singular values on its diagonal, ordered decreasingly.
Let $p=\min(n,m)$ and $d=\operatorname{rank}(Y)$.
Only $d$ singular values are positive in exact arithmetic.

| Representation | Factor dimensions | What is retained? |
| -------------- | ----------------- | ----------------- |
| Full | $n\times n$, $n\times m$, $m\times m$ | Complete orthogonal coordinate systems, including directions unused by the matrix. |
| Economy/thin | $n\times p$, $p\times p$, $m\times p$ | An exact factorization, potentially still including zero singular values. |
| Compact | $n\times d$, $d\times d$, $m\times d$ | Only nonzero singular directions; still exact. |
| Truncated at rank $r<d$ | $n\times r$, $r\times r$, $m\times r$ | An approximation obtained by discarding the smaller singular values. |

For positive singular values, multiplying the decomposition and its transpose gives

$$
Yv_i=\sigma_i u_i,\qquad Y^Tu_i=\sigma_i v_i,
\qquad Y^TYv_i=\sigma_i^2v_i,\quad YY^Tu_i=\sigma_i^2u_i.
$$
The **right** singular vectors describe combinations of input columns; the **left** singular vectors describe directions in the output/state coordinate space.
A unit sphere in the input is first rotated by $V^T$, stretched by the singular values, and oriented in the output by $U$.
Its image is an ellipsoid, possibly lying in a lower-dimensional subspace.

![Unit circle and its image under a matrix](https://feelpp.github.io/course-rom/rom/_images/reduction/svd-geometry.svg)

## From a matrix factorization to modal coefficients

The compact expansion and its individual columns are

$$
Y=\sum_{i=1}^d\sigma_i u_iv_i^T,
\qquad y_j=\sum_{i=1}^d\sigma_i V_{ji}u_i.
$$
Thus the coefficient of column $j$ in direction $u_i$ is $u_i^Ty_j=\sigma_iV_{ji}$.
With all positive modes, the columns are reconstructed exactly; with only the first $r$ modes, their Euclidean projections are $U_rU_r^Ty_j$.
The matrix $U_r^TY$ contains coefficients, while $U_rU_r^TY$ contains reconstructed state vectors. These have different dimensions.
In NumPy, `U, sigma, Vt = np.linalg.svd(Y, full_matrices=False)` returns the economy form: `sigma` is a one-dimensional array and `Vt` is the transpose of the right-vector matrix.
The reconstruction is `(U * sigma) @ Vt`, and rank-$r$ projection coefficients are `U[:, :r].T @ Y`.
Changing the sign of a left singular vector and its matching right vector leaves the matrix unchanged.
Repeated singular values allow rotations within the associated subspace, so compare projectors or spans rather than individual mode signs.
## Truncation, matrix norms and best rank-r approximation

For $Y_r=\sum_{i=1}^r\sigma_i u_iv_i^T$, the two relevant error norms are

$$
\|Y-Y_r\|_2=\sigma_{r+1},\qquad
\|Y-Y_r\|_F^2=\sum_{i>r}\sigma_i^2.
$$
The induced 2-norm is the largest amplification of a unit input vector; the Frobenius norm is the square root of the sum of squared entries, equivalently the sum of squared column lengths.
Taking $r=0$ gives $\|Y\|_2=\sigma_1$ and $\|Y\|_F^2=\sum_i\sigma_i^2$.
The truncated SVD minimizes either error over all matrices of rank at most $r$ (Eckart–Young optimality).
For the 2-norm lower bound, any rank-$r$ competitor $B$ has a nonzero null direction in the span of the first $r+1$ right singular vectors.
Choose that direction with unit length; then $\|(Y-B)x\|_2=\|Yx\|_2\geq\sigma_{r+1}$.
The truncated SVD attains the bound.
For the Frobenius norm, project columns of $Y$ onto the range of a competitor first: this cannot increase their error. The captured-energy argument below proves that the leading left singular space gives the smallest such projection error.
The pseudoinverse is $Y^+=V_d\operatorname{diag}(\sigma_i^{-1})U_d^T$.
It inverts the action on resolved singular directions and gives the minimum-norm least-squares solution.
For full-column-rank $Y$, its 2-norm condition number is $\sigma_1/\sigma_d$.
Near-zero singular values make inversion unstable; truncation for numerical rank must be distinguished from truncation for a desired approximation error.
## Executed example — Compressing the image from the POD slides

The image-compression example in the [SVD/POD slides, pages 18–19](https://feelpp.github.io/course-rom/course-rom/_attachments/lecture-rbobm-beamer-svd-pod.pdf) uses the magic square in Albrecht Dürer&#8217;s *Melencolia I* (1514).
Here we repeat the ranks 1, 20 and 40 with executable Python.
The supplied image is the original-image panel extracted from page 19, converted to grayscale without resizing: 550 rows by 569 columns.
It is a raster reproduction from the slides, not the unavailable original pixel matrix; the numerical errors and storage ratios below are recomputed for this image.
Download [melencolia-magic-square.png](https://feelpp.github.io/course-rom/rom/_attachments/data/melencolia-magic-square.png) beside the notebook before running it.
The code uses NumPy and Matplotlib; no network request is made during execution.
## Pixels are matrix entries

Let $Y_{ij}\in[0,1$] be the gray level at row $i$ and column $j$, with 0 black and 1 white.
Each image column is one snapshot in $\mathbb R^h$; here the metric is Euclidean and no mean is subtracted.
With $Y=U\Sigma V^T$, the first $r$ left singular vectors form the POD basis $Z=U_r$, and

$$
Y_r=U_r\Sigma_rV_r^T=Z(Z^TY),\qquad
\varepsilon_r=\frac{\|Y-Y_r\|_F}{\|Y\|_F}
=\sqrt{\frac{\sum_{i>r}\sigma_i^2}{\sum_i\sigma_i^2}}.
$$
The code computes the SVD once and reuses its factors for every rank.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

image_path = Path('melencolia-magic-square.png')
if not image_path.exists():
    image_path = Path('materials/modules/ROOT/attachments/data/melencolia-magic-square.png')
if not image_path.exists():
    raise FileNotFoundError('Download melencolia-magic-square.png beside this notebook.')
Yimg = plt.imread(image_path).astype(np.float64)
assert Yimg.ndim == 2 and 0 <= Yimg.min() <= Yimg.max() <= 1
height, width = Yimg.shape
img_U, img_sigma, img_Vt = np.linalg.svd(Yimg, full_matrices=False)
img_energy = np.sum(img_sigma**2)
# tails[r] is the squared Frobenius error after retaining r modes.
img_tails = np.r_[np.cumsum((img_sigma**2)[::-1])[::-1], 0.0]
img_relative_errors = np.sqrt(img_tails / img_energy)

def image_at_rank(rank):
    if not 0 <= rank <= len(img_sigma):
        raise ValueError('Rank must lie between zero and min(height, width).')
    return (img_U[:, :rank] * img_sigma[:rank]) @ img_Vt[:rank, :]

print(f'Grayscale image: {height} rows x {width} columns')
print(f'Economy SVD: U {img_U.shape}, sigma {img_sigma.shape}, Vt {img_Vt.shape}')


## Compare the original with ranks 1, 20 and 40

All four panels use the same gray scale.
A truncated reconstruction can have values outside $\lbrack0,1\rbrack$; clipping is used **only for display**.
The error calculations use the unmodified matrix $Y_r$, to which the SVD tail identity applies.


In [ ]:
image_ranks = [1, 20, 40]  # Try 5, 10 or 80, then rerun the following cells.
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
axes[0, 0].imshow(Yimg, cmap='gray', vmin=0, vmax=1)
axes[0, 0].set_title(f'Original: {height} x {width} pixels')
for ax, rank in zip(axes.flat[1:], image_ranks):
    Yr = image_at_rank(rank)
    ax.imshow(np.clip(Yr, 0, 1), cmap='gray', vmin=0, vmax=1)
    ax.set_title(f'Rank {rank}: relative error {img_relative_errors[rank]:.1%}')
for ax in axes.flat:
    ax.axis('off')
fig.tight_layout()
plt.show()


At rank 1, the image is a single outer product: every column is a multiple of one vertical profile.
Increasing rank adds independent profiles, recovering the grid and then smaller details.
This is the same projection mechanism as snapshot POD; image columns replace PDE solution snapshots.
There is no reduced PDE solve or prediction at a new parameter in this example.
## Singular values, error and rank selection

The first singular value is strongly influenced by the uncentered gray-level background.
A high retained-energy percentage therefore does not necessarily preserve visually important digits or fine textures.
For example, 99% retained squared energy allows a 10% relative Frobenius error.
The following calculation selects the smallest rank attaining a chosen **relative error**, rather than confusing energy and error percentages.


In [ ]:
image_tolerance = 0.05
image_rank_for_tolerance = int(np.flatnonzero(img_relative_errors <= image_tolerance)[0])
print(f'Smallest rank for {image_tolerance:.0%} relative Frobenius error: {image_rank_for_tolerance}')
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6))
indices = np.arange(1, len(img_sigma) + 1)
axes[0].semilogy(indices, img_sigma / img_sigma[0], '-')
axes[0].set(xlabel='Singular-value index', ylabel='sigma_i / sigma_1 (dimensionless)')
axes[1].semilogy(np.arange(len(img_sigma)), img_relative_errors[:-1], '-', label='Relative error')
axes[1].plot(image_ranks, img_relative_errors[image_ranks], 'o', label='Displayed ranks')
axes[1].axhline(image_tolerance, linestyle='--', color='black', label='Target error')
axes[1].set(xlabel='Retained rank', ylabel='Relative Frobenius error (dimensionless)')
axes[1].legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()


## What is actually compressed?

Store the three factors, not the dense reconstructed image: $U_r$ has $hr$ entries, $\Sigma_r$ is stored as $r$ numbers and $V_r^T$ has $rw$ entries.
Thus the **fraction of scalar coefficients retained** is $r(h+w+1)/(hw)$.
If singular values are absorbed into one factor, two matrices suffice and this becomes $r(h+w)/(hw)$, the convention used in the slides.
Neither count is a PNG or JPEG file-size ratio.
In particular, storing float64 factors uses eight bytes per scalar, whereas this source image has one byte per grayscale pixel before PNG encoding.
The last column below makes that distinction explicit and excludes file headers and entropy coding.


In [ ]:
print('rank | relative error | retained energy | scalar fraction | float64 / raw uint8 bytes')
for rank in image_ranks:
    Yr = image_at_rank(rank)
    measured = np.linalg.norm(Yimg - Yr, 'fro') / np.linalg.norm(Yimg, 'fro')
    predicted = img_relative_errors[rank]
    assert np.isclose(measured, predicted, rtol=1e-10, atol=1e-12)
    scalar_fraction = rank * (height + width + 1) / (height * width)
    byte_fraction = np.dtype(np.float64).itemsize * scalar_fraction
    retained_energy = 1 - predicted**2
    print(f'{rank:4d} | {measured:14.6%} | {retained_energy:15.6%} | {scalar_fraction:15.4%} | {byte_fraction:10.4f}')
print('Verified: measured reconstruction errors match the singular-value tails.')


**Try it.** Find the smallest rank at which you can read every number, then compare that subjective criterion with the 5% norm tolerance.
Compute the largest rank that saves scalar coefficients, and the largest that saves bytes when using float64 factors instead of a raw uint8 array.
Explain why saving the reconstructed dense matrix or a PNG of it does not implement factor storage.
For a PCA extension, subtract the column mean first, store it separately, and add it back during reconstruction; compare the spectrum and include the mean in the storage budget.

## From PCA in S1 to POD in S3

In the S1 Data Processing course, PCA identifies directions of large variation in a dataset.
In this S3 ROM course, a dataset consists of solution snapshots: each parameter value supplies one observation, and each spatial degree of freedom is a feature.
The familiar SVD now constructs a trial space for a reduced physical model.

| PCA language | POD / ROM language |
| ------------ | ------------------ |
| Observation or sample | Snapshot $\mathbf u(\mu_j)$ |
| Feature | State degree of freedom, with a chosen physical metric |
| Principal direction | POD mode |
| Score | Projection coefficient of a snapshot onto a mode |
| Explained variance | Captured centered snapshot energy in the chosen metric |
| Reconstruction from components | State approximation in a reduced space, possibly plus a mean |

Our snapshot matrix $S$ stores observations in **columns**. A data-processing matrix storing observations in rows is $X=S^T$.
This transpose changes where the principal directions appear in the SVD; it does not change the underlying approximation.
For ordinary centered Euclidean PCA, form

$$
\overline{\mathbf u}=\frac1m\sum_{j=1}^m\mathbf u_j,\qquad
S_c=S-\overline{\mathbf u}\mathbf 1_m^T,
\qquad C=\frac{1}{m-1}S_cS_c^T\quad(m>1).
$$
If $S_c=U_c\Sigma_cV_c^T$, the columns of $U_c$ are principal directions, the eigenvalues of $C$ are $\sigma_{c,i}^2/(m-1)$, and the score matrix is $U_c^TS_c=\Sigma_cV_c^T$.
Using $1/m$ rather than $1/(m-1)$ changes variance normalization, not directions or explained-variance ratios.
The ratio for component $i$ is $\sigma_{c,i}^2/\sum_j\sigma_{c,j}^2$, provided the centered data have nonzero variation.
**Centered Euclidean POD with equal snapshot weights gives the same subspace as ordinary PCA.**
Uncentered POD instead minimizes approximation error of the full states in a linear space through zero. Its singular values describe total snapshot energy, which includes the mean; calling that “explained variance” would be misleading.
A physical POD metric can weight space or variables differently from Euclidean PCA. Standardizing each feature by its sample standard deviation is another modeling choice, not an automatic requirement for PDE states.

## Snapshots, metric and sampling weights

Choose training parameters $\mu_1,\ldots,\mu_m$ and assemble

$$
S=[\mathbf u(\mu_1)\ \cdots\ \mathbf u(\mu_m)]\in\mathbb R^{n\times m}.
$$
Specify a symmetric positive definite state metric $G$ and positive snapshot weights $w_j$.
For example, $G=hI$ approximates an integral norm on the uniform grid in the course practical, and $w_j=1/m$ gives equal sampling weights.
The objective is

$$
\min_{Z^TGZ=I_r}\sum_{j=1}^m w_j\|\mathbf u(\mu_j)-ZZ^TG\mathbf u(\mu_j)\|_G^2.
$$
The projection coefficients use $Z^TG\mathbf u$, not $Z^T\mathbf u$ unless $G=I$.
See [the projection formula](https://feelpp.github.io/course-rom/rom/notation.html#orthogonal-projection).
## Why the first POD mode maximizes captured energy

For one metric-normalized direction $z$, the best coefficient for snapshot $u_j$ is its inner product $z^TGu_j$.
Pythagoras gives

$$
\sum_jw_j\|u_j-z(z^TGu_j)\|_G^2
=\sum_jw_j\|u_j\|_G^2-\sum_jw_j(z^TGu_j)^2.
$$
The first term is independent of the direction. Minimizing error is therefore equivalent to maximizing captured energy.
Factor $G=L^TL$ and write $W=\operatorname{diag}(w_j)$. With $b=Lz$ and $Y=LSW^{1/2}$, the maximized quantity becomes the Rayleigh quotient $b^TYY^Tb$ subject to $b^Tb=1$.
Its maximum is $\sigma_1^2$, attained by the leading eigenvector of $YY^T$, that is, the first left singular vector.
The second direction solves the same maximization subject to orthogonality to the first, and so on.
This connects the variance maximization familiar from PCA to the approximation-error formulation used in POD.

## Constructing the POD basis

Factor $G=L^TL$ and define $W=\operatorname{diag}(w_1,\ldots,w_m)$.
Compute the thin SVD

$$
LSW^{1/2}=U\Sigma V^T,\qquad Z=L^{-1}U_{:,1:r}.
$$
Why this transformation? The [state-coordinate identity](https://feelpp.github.io/course-rom/rom/foundations/riesz.html#metric-coordinates) gives $\|v\|_G=\|Lv\|_2$.
Each column of $LSW^{1/2}$ is therefore a snapshot expressed in Euclidean coordinates and scaled by $\sqrt{w_j}$.
Its squared Euclidean length is the required weighted physical energy.
The weights describe how much each sampled parameter contributes: doubling a weight doubles its squared-error penalty.
Their sum need not be one; normalize them to sum to one when reporting an average.
The SVD finds Euclidean-orthonormal directions in these transformed coordinates.
Mapping back gives

$$
Z^TGZ=U_r^TL^{-T}L^TLL^{-1}U_r=I_r.
$$
Use triangular solves rather than explicitly forming an inverse.
If NumPy returns a Cholesky factor $C$ satisfying $G=CC^T$, take $L=C^T$.
For $G=hI$ and uniform weights, take the SVD of $\sqrt{h/m}S$ and divide the left singular vectors by $\sqrt h$.
*Theorem — POD training error*\
Let $\sigma_1\geq\cdots\geq\sigma_d>0$ be the nonzero singular values of $LSW^{1/2}$.
For $r\leq d$, the minimum weighted sum of squared projection errors is


$$
\sum_{j=1}^m w_j\|\mathbf u(\mu_j)-P_G\mathbf u(\mu_j)\|_G^2
=\sum_{i=r+1}^d\sigma_i^2.
$$
*Proof*\
Set $Y=LSW^{1/2}$ and $B=LZ$, so $B^TB=I_r$.
The weighted objective becomes


$$
\|Y-BB^TY\|_F^2=\|Y\|_F^2-\|B^TY\|_F^2.
$$

Here the Frobenius norm is the square root of the sum of squared matrix entries; the equality is Pythagoras applied column by column.
Write $Y=U\Sigma V^T$ and extend the left singular vectors to an orthonormal basis, with zero singular values if needed.
The captured energy is


$$
\|B^TY\|_F^2=\sum_i\sigma_i^2\|B^Tu_i\|_2^2.
$$

Each weight $\|B^Tu_i\|_2^2$ lies between zero and one, and their sum is $r$.
Since the squared singular values are decreasing, this weighted sum is at most the sum of the first $r$ values.
Choosing $B=U_r$ attains that upper bound.
Subtracting captured energy from total energy proves the tail identity; transforming back preserves the declared weighted error.
This is a statement about the sampled projection problem. It does not bound the maximum error over all parameters, nor the error of a Galerkin solve in the same metric.
## Algorithm — Weighted snapshot POD

Apply the preceding weighted-SVD construction using the metric and snapshot weights already chosen.
The requested rank must lie within the numerically resolved snapshot rank; the discarded singular values quantify the training projection error.

```
\begin{algorithm}
\caption{Weighted POD in a prescribed state metric}
\begin{algorithmic}
\INPUT Snapshots $S\in\mathbb R^{n\times m}$; metric $G$; weights $w_j>0$; rank $r$
\REQUIRE $G$ is symmetric positive definite
\STATE Factor $G=L^TL$ and set $W=\operatorname{diag}(w_1,\ldots,w_m)$
\STATE $Y\leftarrow LSW^{1/2}$
\STATE Compute the thin SVD $Y=U\Sigma V^T$
\STATE Determine the numerical rank $d$ using a declared singular-value threshold
\IF{$r>d$}
  \RETURN Requested rank is not resolved by these snapshots
\ENDIF
\STATE Solve $LZ=U_{:,1:r}$
\STATE $E_{\mathrm{train}}^2\leftarrow\sum_{j=r+1}^{d}\sigma_j^2$
\OUTPUT Basis $Z$ with $Z^TGZ=I_r$, and weighted training error
\end{algorithmic}
\end{algorithm}
```
This is uncentered POD. For centered PCA/POD, subtract the weighted mean first and retain it in the affine reduced model described below.
The training tail is a projection diagnostic, not a held-out Galerkin certificate.
## Method of snapshots: compute modes from correlations

For a PDE, $n$ can be very large while the number of snapshots $m$ is modest.
Rather than factoring a large state metric explicitly, compute the small matrix

$$
C_s=W^{1/2}S^TGSW^{1/2}=Y^TY\in\mathbb R^{m\times m}.
$$
Its entry $(C_s)_{ij}=\sqrt{w_iw_j}\,u_i^TGu_j$ measures similarity of two snapshots in the declared state metric.
It is a Gram/correlation matrix in snapshot coordinates, not the state covariance matrix of centered PCA.
The two are related through the SVD but have different dimensions.
Solve its symmetric eigenproblem with Euclidean-normalized eigenvectors:

$$
C_sv_i=\lambda_i v_i,\qquad v_i^Tv_j=\delta_{ij},\qquad
\lambda_i=\sigma_i^2.
$$
For each retained positive eigenvalue, reconstruct the physical mode:

$$
z_i=\frac{SW^{1/2}v_i}{\sqrt{\lambda_i}}.
$$
The factor $W^{1/2}$ is essential when weights are not all one.
The identity follows by inserting $u_i=Yv_i/\sigma_i$ in $z_i=L^{-1}u_i$.
It also proves metric orthonormality directly:

$$
z_i^TGz_j=\frac{v_i^TC_sv_j}{\sqrt{\lambda_i\lambda_j}}=\delta_{ij}.
$$
No square root of the full metric is needed in this construction: use matrix products with $G$, which can remain sparse.
For equal averaging weights $1/m$, form $C_s=S^TGS/m$ and normalize modes by $\sqrt{m\lambda_i}$.
The eigenvalue tail then measures a **mean** squared error, not an unnormalized sum.
## Which computation should be used?


- If $m\ll n$, the $m\times m$ snapshot matrix can be much smaller than a state-space covariance matrix.
- If $n\ll m$, forming an $m\times m$ correlation matrix is not the dimensional saving. Use a direct economy SVD or the smaller transformed state covariance $YY^T$.
- Forming a Gram matrix squares the ratio of largest to smallest resolved singular values. A direct SVD generally resolves small modes more reliably; do not divide by an unresolved eigenvalue.

For dense arrays, a tall-matrix SVD costs on the order of $nm^2$ when $m\leq n$.
Correlation assembly costs products with $G$ plus roughly $nm^2$ work, followed by an $m\times m$ eigensolve; its advantage is the small eigenproblem and avoiding a full metric factorization, not removal of all full-dimensional offline work.
## Algorithm — Weighted correlation POD

Use a numerical eigenvalue threshold first, then select a physical approximation rank.
If the requested tolerance depends on unresolved modes, report that limitation rather than silently promising it has been met.

```
\begin{algorithm}
\caption{POD from weighted snapshot correlations}
\begin{algorithmic}
\INPUT Snapshots $S$; metric $G$; positive weights $W$; rank $r$; eigenvalue threshold $\tau$
\STATE $S_w\leftarrow SW^{1/2}$; $C_s\leftarrow S_w^TGS_w$
\STATE Solve the symmetric eigenproblem and sort eigenpairs in decreasing order
\STATE Check that negative eigenvalues are within declared roundoff tolerance
\STATE $d\leftarrow$ number of eigenvalues greater than $\tau$
\IF{$r>d$}
  \RETURN Requested rank is not numerically resolved
\ENDIF
\FOR{$i=1$ \TO $r$}
  \STATE $z_i\leftarrow S_wv_i/\sqrt{\lambda_i}$
\ENDFOR
\STATE $Z\leftarrow[z_1,\ldots,z_r]$
\STATE Check $Z^TGZ$ and the snapshot projection error
\OUTPUT Basis $Z$ and the resolved eigenvalue spectrum
\end{algorithmic}
\end{algorithm}
```
## Worked computation with a non-diagonal metric

The following example compares the correlation result with a direct SVD, using unequal weights and a metric that is not diagonal.
It compares projectors because singular vectors may differ by signs.


In [ ]:
import numpy as np

S = np.array([[1., 0.], [0., 2.], [1., 1.]])
G = np.array([[2., 1., 0.], [1., 2., 0.], [0., 0., 3.]])
w = np.array([0.25, 0.75])
Sw = S * np.sqrt(w)
Cs = Sw.T @ G @ Sw
lam, V = np.linalg.eigh(Cs)
order = np.argsort(lam)[::-1]
lam, V = lam[order], V[:, order]
r = 1
Zcorr = (Sw @ V[:, :r]) / np.sqrt(lam[:r])
L = np.linalg.cholesky(G).T
U, sigma, Vt = np.linalg.svd(L @ Sw, full_matrices=False)
Zsvd = np.linalg.solve(L, U[:, :r])
assert np.allclose(lam, sigma**2)
assert np.allclose(Zcorr.T @ G @ Zcorr, np.eye(r))
assert np.allclose(Zcorr @ Zcorr.T @ G, Zsvd @ Zsvd.T @ G)
E = S - Zcorr @ (Zcorr.T @ G @ S)
error_squared = np.sum(w * np.sum(E * (G @ E), axis=0))
assert np.isclose(error_squared, np.sum(lam[r:]))
print("Correlation POD: weighted projection error squared =", error_squared)


## Choosing and testing a rank

For nonzero snapshots, a relative training criterion is

$$
\frac{\sum_{i>r}\sigma_i^2}{\sum_i\sigma_i^2}\leq\varepsilon_{\mathrm{POD}}^2.
$$
Equivalently, the **relative information content** (RIC), or captured-energy fraction, is

$$
\mathrm{RIC}(r)=\frac{\sum_{i=1}^r\sigma_i^2}{\sum_i\sigma_i^2}
\geq1-\varepsilon_{\mathrm{POD}}^2.
$$
The square matters: the tail ratio is a relative squared error.
For example, singular values $(4,2,1)$ have total energy 21. Ranks one and two capture $16/21$ and $20/21$, respectively.
Capturing at least 95% selects rank two, but the relative RMS error is still $\sqrt{1/21}\simeq0.218$.
Capturing 99% energy means a relative RMS error of at most 10%, not 1%.
For zero snapshots, total energy is zero and the ratio is undefined; the zero space already reconstructs the data exactly.
Do not retain numerically meaningless singular vectors merely to reach a requested rank.
Then test unseen parameters and separately report:

- Projection error in the declared state metric.
- Error of the reduced solve in that metric.
- Error in the required outputs or observations.

Training only where solutions are similar may miss another regime. A posteriori estimation and greedy selection provide complementary information.
Centering snapshots changes the approximation to an affine space $\overline{\mathbf u}+\operatorname{range}(Z)$; the reduced equations must include the mean contribution. The first practical uses uncentered snapshots.

## A centered PCA space needs an affine reduced model

For unequal sampling weights, the mean minimizing the weighted squared distance to a constant state is

$$
\overline{\mathbf u}=\frac{\sum_j w_j\mathbf u_j}{\sum_j w_j}.
$$
Differentiating that quadratic objective gives this formula; the fixed positive definite metric cancels from its normal equations.
Subtract this same mean from every snapshot before the weighted SVD.
For centered modes, reconstruct $\mathbf u_r=\overline{\mathbf u}+Z\mathbf a$.
Galerkin testing gives

$$
Z^TA(\mu)Z\mathbf a=Z^T\bigl(\mathbf f(\mu)-A(\mu)\overline{\mathbf u}\bigr).
$$
For the course&#8217;s affine operator, precompute $Z^TK\overline{\mathbf u}$ and $Z^T\overline{\mathbf u}$ in addition to the usual reduced operators.
The output also retains its mean contribution: $s_r=\mathbf l^T\overline{\mathbf u}+(Z^T\mathbf l)^T\mathbf a$.
Inserting centered modes into the uncentered equations and forgetting the mean solves a different approximation problem.
The SVD gives projection scores for available snapshots. At a new parameter, Galerkin determines coefficients from the governing equations without requiring a full-order snapshot.
This is the key step from a data reconstruction technique to a predictive ROM.

## Supplementary reading — Continuous POD and the meaning of sampling

A snapshot objective is a finite approximation to a declared objective over parameter space.
Let $\nu$ be a measure on $\mathcal P$: it can describe a physical parameter distribution or normalized volume over an operating range.
Assume square-integrability of the solution map:

$$
\int_{\mathcal P}\|u_h(\mu)\|_V^2\,d\nu(\mu)<\infty.
$$
The continuous rank-$r$ POD problem minimizes

$$
\int_{\mathcal P}\|u_h(\mu)-P_{V_r}u_h(\mu)\|_V^2\,d\nu(\mu)
$$
over subspaces of dimension $r$, with orthogonal projection in the declared state inner product.
It is an average criterion; minimizing it does not minimize the worst parameterwise error.
## The snapshot operator and its adjoint

A continuous analogue of the snapshot matrix is the map $T:L^2(\mathcal P,\nu)\to V_h$ defined by

$$
Tg=\int_{\mathcal P}u_h(\mu)g(\mu)\,d\nu(\mu).
$$
Its adjoint is characterized by $(Tg,v)_V=(g,T^*v)_{L^2(\nu)}$.
Substituting the integral yields

$$
(T^*v)(\mu)=(u_h(\mu),v)_V.
$$
Thus $T^*$ extracts the coefficient function of a state direction over parameter space.
The state correlation operator $\mathcal K=TT^*$ is

$$
\mathcal Kv=\int_{\mathcal P}u_h(\mu)(u_h(\mu),v)_V\,d\nu(\mu).
$$
It is self-adjoint and positive semidefinite: its quadratic form is the integral of squared coefficients.
It need not be positive definite if the solution family misses a state direction.
Its metric-orthonormal eigenvectors satisfy $\mathcal Kz_i=\lambda_i z_i$, with nonnegative decreasing eigenvalues.
The parameter-space operator $\mathcal C=T^*T$ instead has kernel $(u_h(\mu),u_h(\mu'))_V$, the continuous analogue of snapshot correlations.
## Optimal modes and the continuous tail

For a positive eigenvalue, define $\psi_i=T^*z_i/\sqrt{\lambda_i}$.
Then $\psi_i$ are orthonormal in parameter space, $T\psi_i=\sqrt{\lambda_i}z_i$, and the state has the expansion

$$
u_h(\mu)=\sum_{i=1}^d\sqrt{\lambda_i}z_i\psi_i(\mu)
\quad\text{in }L^2(\mathcal P,\nu;V_h),\qquad d\leq n.
$$
The first $r$ eigenvectors minimize the integrated squared projection error, with minimum $\sum_{i>r}\lambda_i$.
To see this, for any orthonormal trial directions $b_j$, the captured energy is $\sum_{j=1}^r(\mathcal Kb_j,b_j)_V$.
Expand each direction in the eigenbasis: as in the discrete POD proof, the largest possible sum is the sum of the first $r$ eigenvalues.
Subtract it from the total energy, which is $\sum_i\lambda_i$.
The operator formulation expresses the same result using the Hilbert–Schmidt norm:

$$
\|T\|_{\mathrm{HS}}^2=\int_{\mathcal P}\|u_h(\mu)\|_V^2\,d\nu(\mu),
\qquad
\min_{\operatorname{rank}(B)\leq r}\|T-B\|_{\mathrm{HS}}^2
=\sum_{i>r}\lambda_i.
$$
The minimizing operator is $P_{V_r}T$, the truncated singular expansion (Schmidt approximation).
Here the target space is finite dimensional, so this spectral argument requires no infinite sequence of state modes.
For centered random states with a probability measure, this is also the covariance/Karhunen–Loève interpretation of POD.
## Return to a finite sample: quadrature and probability

Choose positive quadrature weights and parameter points such that

$$
\int_{\mathcal P}F(\mu)\,d\nu(\mu)\approx\sum_{j=1}^m w_j F(\mu_j).
$$
Applying this rule to squared projection errors gives exactly the weighted snapshot objective and correlation matrix $W^{1/2}S^TGSW^{1/2}$.
The sampling design and the weights together specify which continuous average is being approximated.

- Independent samples drawn from the desired probability distribution use equal Monte Carlo weights $1/m$.
- A uniform grid with a trapezoidal integration rule gives half-weight to the interval endpoints; normalize by interval length for a uniform probability average.
- Equal weights on samples uniform in $t=\log\mu$ approximate a log-uniform average. For a uniform physical-parameter average, the substitution $d\mu=e^t dt$ introduces the Jacobian into quadrature weights.

Increasing the snapshot count can improve this integral approximation, but the finite-data singular-value tail measures only the discrete training error.
It is not an estimate of quadrature error and does not prove a uniform error bound over the entire parameter domain.
Compare independent validation/test samples and use residual bounds for the reduced solves when their assumptions hold.
## Approximation rank, denoising and compression

Energy truncation, numerical-rank detection and denoising answer different questions.
When snapshots are noisy, a mode can contain noise energy; retaining a chosen fraction of total energy does not by itself distinguish signal from noise.
A statistical singular-value threshold requires assumptions on the noise distribution and scaling, which need not hold for deterministic PDE discretization errors.
The S1 image-compression analogy is useful: storing a rank-$r$ factorization needs approximately $r(n+m+1)$ numbers instead of $nm$, but small pixel reconstruction error does not certify a physical PDE output.
The course uses noiseless computed snapshots for the basic POD construction and introduces explicit observation noise in the assimilation units.

## Exercises before running the practical

Take two snapshots $\mathbf u_1=(1,0)^T$ and $\mathbf u_2=(0,2)^T$, equal weights and $G=I$.
Identify a rank-one POD space and its mean squared projection error.
Repeat with $G=\operatorname{diag}(9,1)$. Explain why the preferred direction changes.

1. Derive the identity between singular values and correlation eigenvalues, including dimensions.
1. Compare SVD and correlation POD with unequal weights using the worked Python example.
1. For the spectrum $(4,2,1)$, choose ranks for a 95% captured-energy target and a 5% relative RMS-error target.
1. Explain which average is approximated by equal weights on linearly spaced versus logarithmically spaced parameters.

The continuous operator proof is supplementary reading; it is not an additional required implementation in the two-hour session.
Continue with the [executable practical](https://feelpp.github.io/course-rom/course-rom/labs/session02-galerkin-pod.html) and [follow-up exercises](https://feelpp.github.io/course-rom/course-rom/homework/01-reduced-models.html).

## References and further reading

For the connection between POD and SVD, see Section 1.1 of [#volkwein-pod](https://feelpp.github.io/course-rom/rom/reduction/pod.html#volkwein-pod).

- [undefined] S. Volkwein, [*Model Reduction using Proper Orthogonal Decomposition* (PDF)](https://www.math.uni-konstanz.de/numerik/personen/volkwein/teaching/POD-Vorlesung.pdf), Universität Konstanz lecture notes, 7 December 2011.
